# 07 — Gold Dimensions

**Purpose:** Create Gold dimension tables from the cleaned Silver master tables.


In [0]:
from pyspark.sql.functions import col

SILVER = "capstone_team01.silver_schema"
GOLD = "capstone_team01.gold_schema"

spark.sql("CREATE SCHEMA IF NOT EXISTS capstone_team01.gold_schema")


## 1. Customer Dimension


In [0]:
customer = spark.table(f"{SILVER}.silver_customer_master")

dim_customer = customer.select(
    "customer_id",
    "customer_segment",
    "onboarded_date",
    "home_country",
    "account_status",
    "kyc_status",
    "risk_band",
    "relationship_manager",
    "city",
    "customer_type"
)

dim_customer.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{GOLD}.dim_customer")

print("Customer dimension count:", dim_customer.count())
display(dim_customer.limit(5))


## 2. Merchant Dimension


In [0]:
merchant = spark.table(f"{SILVER}.silver_merchant_master")

dim_merchant = merchant.select(
    "merchant_id",
    "merchant_name",
    "merchant_category",
    "merchant_country",
    "onboarded_date",
    "merchant_status",
    "settlement_currency",
    "risk_band",
    "city",
    "acquirer"
)

dim_merchant.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{GOLD}.dim_merchant")

print("Merchant dimension count:", dim_merchant.count())
display(dim_merchant.limit(5))


## 3. Card Dimension


In [0]:
card = spark.table(f"{SILVER}.silver_card_master")

dim_card = card.select(
    "card_id",
    "customer_id",
    "card_type",
    "issuer_country",
    "issue_date",
    "expiry_date",
    "card_status",
    "network",
    "daily_limit",
    "tokenized_flag"
)

dim_card.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{GOLD}.dim_card")

print("Card dimension count:", dim_card.count())
display(dim_card.limit(5))


## 4. Final Validation


In [0]:
print("Dim Customer:", spark.table(f"{GOLD}.dim_customer").count())
print("Dim Merchant:", spark.table(f"{GOLD}.dim_merchant").count())
print("Dim Card:", spark.table(f"{GOLD}.dim_card").count())

print("Duplicate customer IDs:",
      spark.table(f"{GOLD}.dim_customer")
      .groupBy("customer_id").count()
      .filter(col("count") > 1).count())

print("Duplicate merchant IDs:",
      spark.table(f"{GOLD}.dim_merchant")
      .groupBy("merchant_id").count()
      .filter(col("count") > 1).count())

print("Duplicate card IDs:",
      spark.table(f"{GOLD}.dim_card")
      .groupBy("card_id").count()
      .filter(col("count") > 1).count())
